# Exploratory Data Analysis

This notebook joins the menu data with the daily statistics at the correct grain, then explores the merged dataset.

## Join strategy

- `menu_data.email` behaves like a `dou_code` key rather than a personal email.
- `statistics_dataa` is at restaurant level, so several rows can exist for the same `dou_code + date`.
- `menu_data` is effectively one daily menu per `dou_code + date`, repeated across menu items.
- Because of that, the safe join is:
  1. clean and normalize the menu dates,
  2. aggregate `statistics_dataa` and `client_data` to `dou_code + date`,
  3. join the aggregated statistics back to the menu daily table.


In [2]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.options.display.float_format = '{:,.2f}'.format
plt.style.use('ggplot')


In [4]:
menu_data = pd.read_excel('../data/raw/Menu_20260411_anonymized.xlsx')
statistics_dataa = pd.read_excel('../data/raw/Statistics_per_day_20260410_anonymized.xlsx')
client_data = pd.read_excel('../data/raw/Statistics_per_day_type_client_20260410_anonymized.xlsx')

print('menu_data:', menu_data.shape)
print('statistics_dataa:', statistics_dataa.shape)
print('client_data:', client_data.shape)


menu_data: (570803, 7)
statistics_dataa: (249837, 9)
client_data: (250249, 21)


In [5]:
YEAR_FIX_MAP = {
    '0002': 2023,
    '0004': 2024,
    '0006': 2026,
    '0020': 2024,
    '0023': 2023,
    '0024': 2024,
    '0025': 2025,
    '0026': 2026,
    '0202': 2024,
    '0203': 2024,
    '0204': 2024,
    '0205': 2025,
    '0206': 2026,
    '0224': 2024,
    '0225': 2025,
    '0242': 2024,
}

MEAL_TYPE_LABELS = {
    1: 'breakfast',
    2: 'lunch',
    3: 'dinner',
}


def repair_menu_date(value):
    if pd.isna(value):
        return pd.NaT

    if isinstance(value, pd.Timestamp):
        return value.normalize()

    parsed = pd.to_datetime(value, errors='coerce')
    if pd.notna(parsed) and 2023 <= parsed.year <= 2026:
        return parsed.normalize()

    text = str(value).strip()
    match = re.fullmatch(r'(\d{4})-(\d{2})-(\d{2})', text)
    if not match:
        return pd.NaT

    year_token, month, day = match.groups()
    fixed_year = YEAR_FIX_MAP.get(year_token)

    if fixed_year is None:
        embedded_years = [yy for yy in ('23', '24', '25', '26') if yy in year_token]
        if len(embedded_years) == 1:
            fixed_year = int(f'20{embedded_years[0]}')

    if fixed_year is None:
        return pd.NaT

    return pd.Timestamp(f'{fixed_year}-{month}-{day}')


menu_clean = (
    menu_data.assign(
        dou_code=pd.to_numeric(menu_data['email'], errors='coerce').astype('Int64'),
        date=menu_data['date'].apply(repair_menu_date),
        meal=menu_data['meal'].astype(str).str.strip(),
        meal_type_id=pd.to_numeric(menu_data['meal_type_id'], errors='coerce').astype('Int64'),
    )
    .dropna(subset=['dou_code', 'date'])
)

statistics_clean = statistics_dataa.assign(
    dou_code=pd.to_numeric(statistics_dataa['dou_code'], errors='coerce').astype('Int64'),
    date=pd.to_datetime(statistics_dataa['create_date'], errors='coerce').dt.normalize(),
)

client_clean = client_data.assign(
    dou_code=pd.to_numeric(client_data['dou_code'], errors='coerce').astype('Int64'),
    date=pd.to_datetime(client_data['create_date'], errors='coerce').dt.normalize(),
)

print('Unresolved menu dates after repair:', menu_clean['date'].isna().sum())
display(menu_clean.head())


Unresolved menu dates after repair: 0


,meal,quantity,date,meal_type_id,name,email,id,dou_code
0,كاس حليب,1,2023-11-05,1,ANON_16ed62abf995,11,1,11
1,مادلين,2,2023-11-05,1,ANON_16ed62abf995,11,1,11
2,كرواسون,1,2023-11-05,1,ANON_16ed62abf995,11,1,11
3,فيطاجي,1,2023-11-05,2,ANON_16ed62abf995,11,1,11
4,عدس,200 غرام,2023-11-05,2,ANON_16ed62abf995,11,1,11


## Build daily DOU-level tables

The menu file is item-level, while the statistics file is restaurant-level. Aggregating both statistics tables to `dou_code + date` prevents row explosion and keeps the join meaningful.


In [6]:
menu_daily = (
    menu_clean.groupby(['dou_code', 'date'], as_index=False)
    .agg(
        menu_name=('name', 'first'),
        menu_source_id=('id', 'first'),
        menu_item_rows=('meal', 'size'),
        unique_meals=('meal', 'nunique'),
        breakfast_items=('meal_type_id', lambda s: (s == 1).sum()),
        lunch_items=('meal_type_id', lambda s: (s == 2).sum()),
        dinner_items=('meal_type_id', lambda s: (s == 3).sum()),
    )
)

statistics_daily = (
    statistics_clean.groupby(['dou_code', 'date'], as_index=False)
    .agg(
        restaurant_count=('resto_name', 'nunique'),
        total_count=('count', 'sum'),
        breakfast_count=('breakfast', 'sum'),
        lunch_count=('launch', 'sum'),
        dinner_count=('dinner', 'sum'),
    )
)

client_daily = (
    client_clean.groupby(['dou_code', 'date'], as_index=False)
    .agg(
        client_count_total=('count', 'sum'),
        breakfast_etudiant=('breakfast_etudiant', 'sum'),
        lunch_etudiant=('launch_etudiant', 'sum'),
        dinner_etudiant=('dinner_etudiant', 'sum'),
        breakfast_employe=('breakfast_employe', 'sum'),
        lunch_employe=('launch_employe', 'sum'),
        dinner_employe=('dinner_employe', 'sum'),
        breakfast_doctorant=('breakfast_doctorant', 'sum'),
        lunch_doctorant=('launch_doctorant', 'sum'),
        dinner_doctorant=('dinner_doctorant', 'sum'),
        breakfast_para_medical=('breakfast_para_medical', 'sum'),
        lunch_para_medical=('launch_para_medical', 'sum'),
        dinner_para_medical=('dinner_para_medical', 'sum'),
    )
)

joined_df = (
    menu_daily.merge(statistics_daily, on=['dou_code', 'date'], how='inner')
    .merge(client_daily, on=['dou_code', 'date'], how='left')
    .sort_values(['date', 'dou_code'])
    .reset_index(drop=True)
)

reachable_canteens = (
    statistics_clean.merge(joined_df[['dou_code', 'date']].drop_duplicates(), on=['dou_code', 'date'], how='inner')
    ['resto_name']
    .nunique()
)

coverage_summary = pd.Series({
    'menu_dou_days': len(menu_daily),
    'statistics_dou_days': len(statistics_daily),
    'client_dou_days': len(client_daily),
    'joined_dou_days': len(joined_df),
    'menu_join_rate_pct': len(joined_df) / len(menu_daily) * 100,
    'statistics_join_rate_pct': len(joined_df) / len(statistics_daily) * 100,
    'linked_dous': joined_df['dou_code'].nunique(),
    'shared_dates': joined_df['date'].nunique(),
    'reachable_canteens': reachable_canteens,
    'stats_only_dous': ', '.join(map(str, sorted(set(statistics_daily['dou_code']) - set(menu_daily['dou_code'])))),
})

display(coverage_summary.to_frame('value'))
display(joined_df.head())


,value
menu_dou_days,40328
statistics_dou_days,40399
client_dou_days,40465
joined_dou_days,35931
menu_join_rate_pct,89.10
statistics_join_rate_pct,88.94
linked_dous,66
shared_dates,734
reachable_canteens,506
stats_only_dous,1601


,dou_code,date,menu_name,menu_source_id,menu_item_rows,unique_meals,breakfast_items,lunch_items,dinner_items,restaurant_count,total_count,breakfast_count,lunch_count,dinner_count,client_count_total,breakfast_etudiant,lunch_etudiant,dinner_etudiant,breakfast_employe,lunch_employe,dinner_employe,breakfast_doctorant,lunch_doctorant,dinner_doctorant,breakfast_para_medical,lunch_para_medical,dinner_para_medical
0,41,2023-12-12,ANON_47514afc3ea2,4,11,9,2,5,4,1,6,0,6,0,6,0,6,0,0,0,0,0,0,0,0,0,0
1,52,2023-12-12,ANON_129d1f8c0796,6,11,11,2,4,5,1,1,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0
2,92,2023-12-12,ANON_26fc6f268975,14,13,11,2,6,5,1,65,0,65,0,65,0,64,0,0,1,0,0,0,0,0,0,0
3,121,2023-12-12,ANON_da4145ff4778,17,13,12,3,5,5,1,1,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0
4,141,2023-12-12,ANON_8e5803b6bcae,20,12,10,2,5,5,1,2,0,2,0,2,0,2,0,0,0,0,0,0,0,0,0,0


In [7]:
joined_df = joined_df.assign(
    weekday=joined_df['date'].dt.day_name(),
    month=joined_df['date'].dt.to_period('M').astype(str),
    student_total=joined_df[['breakfast_etudiant', 'lunch_etudiant', 'dinner_etudiant']].sum(axis=1),
    employe_total=joined_df[['breakfast_employe', 'lunch_employe', 'dinner_employe']].sum(axis=1),
    doctorant_total=joined_df[['breakfast_doctorant', 'lunch_doctorant', 'dinner_doctorant']].sum(axis=1),
    para_medical_total=joined_df[['breakfast_para_medical', 'lunch_para_medical', 'dinner_para_medical']].sum(axis=1),
)

quality_checks = pd.Series({
    'min_date': joined_df['date'].min(),
    'max_date': joined_df['date'].max(),
    'missing_values_total': int(joined_df.isna().sum().sum()),
    'client_count_matches_statistics': bool(np.allclose(joined_df['total_count'], joined_df['client_count_total'])),
})

numeric_summary = joined_df[
    ['total_count', 'breakfast_count', 'lunch_count', 'dinner_count', 'restaurant_count', 'menu_item_rows', 'unique_meals']
].describe().T

correlation_summary = pd.Series({
    'corr(total_count, restaurant_count)': joined_df['total_count'].corr(joined_df['restaurant_count']),
    'corr(total_count, menu_item_rows)': joined_df['total_count'].corr(joined_df['menu_item_rows']),
    'corr(total_count, unique_meals)': joined_df['total_count'].corr(joined_df['unique_meals']),
    'share_of_days_with_multiple_restaurants_pct': (joined_df['restaurant_count'].gt(1).mean() * 100),
})

display(quality_checks.to_frame('value'))
display(numeric_summary)
display(correlation_summary.to_frame('value'))


,value
min_date,2023-12-12 00:00:00
max_date,2026-04-09 00:00:00
missing_values_total,0
client_count_matches_statistics,True


,count,mean,std,min,25%,50%,75%,max
total_count,"35,931.00","5,284.57","4,611.07",1.00,"1,577.00","4,145.00","7,703.50","24,902.00"
breakfast_count,"35,931.00",666.91,787.41,0.00,34.00,377.00,"1,030.00","4,885.00"
lunch_count,"35,931.00","2,492.98","2,077.41",0.00,813.00,"2,015.00","3,646.50","11,331.00"
dinner_count,"35,931.00","2,124.67","1,994.17",0.00,544.00,"1,616.00","3,120.50","12,158.00"
restaurant_count,"35,931.00",6.47,3.41,1.00,4.00,7.00,9.00,21.00
menu_item_rows,"35,931.00",14.35,3.94,1.00,12.00,15.00,16.00,58.00
unique_meals,"35,931.00",12.27,2.83,1.00,11.00,12.00,14.00,34.00


,value
"corr(total_count, restaurant_count)",0.75
"corr(total_count, menu_item_rows)",0.12
"corr(total_count, unique_meals)",0.09
share_of_days_with_multiple_restaurants_pct,91.62
